# Building an AI Data Analyst

## Velocity Motors: Multi-Agent Orchestration with Databricks Genie

Velocity Motors is a synthetic automotive dealership dataset with:
- **50,000 customers** across consumer and fleet segments
- **100,000+ orders** with full payment and delivery tracking
- **5,000 vehicles** across 30+ makes and models
- **16 interconnected tables** spanning sales, inventory, service, and HR

In this notebook we build a **multi-agent AI data analyst** that:
- Translates natural language into SQL via **Databricks Genie**
- Retrieves company policies and documents via **RAG** (Retrieval Augmented Generation)
- Routes questions to the right agent via a **LangGraph Supervisor**

## Architecture

![Supervisor Agent Architecture](diagrams/supervisor_agent_diagram.png)

**Genie Agent:** Connects to a Databricks Genie Space backed by Unity Catalog tables.
Translates natural language to SQL, executes it, and returns structured results.

**RAG Agent:** Searches a vector index of company documents (policies, procedures, guidelines).
Retrieves relevant chunks and synthesizes an answer grounded in source documents.

**Supervisor:** An LLM with tool-calling that decides which agent(s) to invoke for each question.

### Velocity Motors Schema (16 tables)

| Sales & Orders | Inventory & Vehicles | Service & Parts | HR & Operations |
|---|---|---|---|
| customers | vehicles | service_records | employees |
| orders | vehicle_features | parts | departments |
| order_items | inventory | parts_suppliers | employee_performance |
| payments | | service_parts | salesperson_quotas |
| leads | | | employee_certifications |
| customer_interactions | | | |

In [ ]:
# Install dependencies (run once)
%pip install databricks-sdk>=0.40.0 databricks-langchain>=0.13.0 databricks-vectorsearch>=0.64 langgraph>=0.2.0 langchain-core>=0.3.0 langchain-text-splitters>=0.3.0 pydantic>=2.0.0 python-dotenv>=1.0.0 -q

In [ ]:
# Restart Python to pick up new packages (Databricks)
dbutils.library.restartPython()

In [ ]:
# Add src to path for imports
import os
import sys

# For Databricks notebooks
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    # Get the workspace path
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    # For local development
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

In [ ]:
# Create Databricks widgets for interactive configuration
import os

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_DATABRICKS:
    # Remove existing widgets to avoid duplicates
    try:
        dbutils.widgets.removeAll()
    except:
        pass

    # === Configuration Widgets ===
    dbutils.widgets.text("genie_space_id", "", "1. Genie Space ID")
    dbutils.widgets.text("warehouse_id", "", "2. Warehouse ID")
    dbutils.widgets.dropdown(
        "model_endpoint",
        "databricks-meta-llama-3-3-70b-instruct",
        [
            "databricks-meta-llama-3-3-70b-instruct",
            "databricks-meta-llama-3-1-405b-instruct",
            "databricks-dbrx-instruct",
        ],
        "3. Model Endpoint",
    )
    dbutils.widgets.dropdown("mock_mode", "false", ["true", "false"], "4. Mock Mode")
    dbutils.widgets.text("vector_search_endpoint", "workshop.rag.document_chunks", "5. Vector Search Endpoint")
    dbutils.widgets.text("vector_search_index", "workshop.rag.document_index", "6. Vector Search Index")

    print("Widgets created! Configure settings using the dropdowns at the top of the notebook.")
    print("\nFor Mock Mode demo, leave Space ID empty and set Mock Mode = true")
else:
    print("Running locally - using environment variables for configuration")
    print("Set: MOCK_MODE=true for demo without Genie access")

In [ ]:
from src.config import Config, clear_config_cache

# Clear any cached config
clear_config_cache()

# Read configuration from widgets (Databricks) or environment variables (local)
if IN_DATABRICKS:
    # Read from Databricks widgets
    genie_space_id = dbutils.widgets.get("genie_space_id") or os.getenv("GENIE_SPACE_ID", "")
    warehouse_id = dbutils.widgets.get("warehouse_id") or os.getenv("WAREHOUSE_ID", "")
    model_endpoint = dbutils.widgets.get("model_endpoint")
    mock_mode = dbutils.widgets.get("mock_mode") == "true"
    vector_search_endpoint = dbutils.widgets.get("vector_search_endpoint") or os.getenv("VECTOR_SEARCH_ENDPOINT", "")
    vector_search_index = dbutils.widgets.get("vector_search_index") or os.getenv("VECTOR_SEARCH_INDEX", "")
else:
    # Read from environment variables (local development)
    genie_space_id = os.getenv("GENIE_SPACE_ID", "")
    warehouse_id = os.getenv("WAREHOUSE_ID", "")
    model_endpoint = os.getenv("MODEL_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")
    mock_mode = os.getenv("MOCK_MODE", "false").lower() == "true"
    vector_search_endpoint = os.getenv("VECTOR_SEARCH_ENDPOINT", "")
    vector_search_index = os.getenv("VECTOR_SEARCH_INDEX", "")

config = Config(
    genie_space_id=genie_space_id,
    warehouse_id=warehouse_id,
    model_endpoint=model_endpoint,
    mock_mode=mock_mode,
    vector_search_endpoint=vector_search_endpoint,
    vector_search_index=vector_search_index,
)

# Validate configuration
errors = config.validate()
if errors:
    print("Configuration warnings:")
    for error in errors:
        print(f"  - {error}")
else:
    print("Configuration valid!")

# Check RAG configuration
if config.is_rag_configured():
    print("RAG (Vector Search) configured!")
else:
    print("RAG running in mock mode (Vector Search not configured)")

print(f"\nGenie Space ID: {config.genie_space_id or '(not set)'}")
print(f"Mock mode: {config.mock_mode}")
print(f"Model endpoint: {config.model_endpoint}")

## Initialize Agents

Create the supervisor with its two tools: `query_data` (Genie) and `search_documents` (RAG).

In [ ]:
from src.agents import RAGAgent
from src.agents.supervisor import create_simple_supervisor

# Initialize the supervisor (includes Genie and RAG agents)
supervisor = create_simple_supervisor(config)

print("Supervisor agent initialized!")
print(f"  - Genie Agent: {'Mock' if config.mock_mode else 'Live'}")
print(f"  - RAG Agent: {'Live (Vector Search)' if config.is_rag_configured() else 'Mock'}")

## Simple Queries: Single Table, Basic Aggregations

These questions target a single table with straightforward aggregation. Genie translates them to simple `SELECT ... GROUP BY` statements.

In [ ]:
# Q1: Simple count, single table, no joins
question = "How many customers do we have in total?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

In [ ]:
# Q6: Basic aggregation, average over one column
question = "What is the average MSRP of all vehicles?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

## Business Logic Queries: Domain Understanding Required

These require understanding business concepts like "revenue" (price x quantity), "conversion rate" (leads that became orders), and how tables join together.

In [ ]:
# Q21: Revenue by payment method, needs orders + payments join
question = "What is the total revenue from completed orders by payment method?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

In [ ]:
# Q24: Lead conversion, needs leads + orders relationship
question = "What is the lead conversion rate by source?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

## Activity 1: Pick a Question

Choose a question from the menu below, or write your own. These cover categories we haven't tested yet: temporal trends, inventory, and multi-table joins.

In [ ]:
# Pick a letter (A-F) or write your own question
questions = {
    "A": "What are the monthly sales trends for the past 12 months?",
    "B": "Which vehicle models have been in inventory the longest?",
    "C": "What is the average discount percentage by customer segment?",
    "D": "Show the top 10 customers by lifetime order value",
    "E": "What is the breakdown of orders by payment method?",
    "F": "Which departments have the most certified employees?",
}

# === CHANGE THIS ===
choice = "A"  # Pick A-F, or set to None and fill in your_question
your_question = None  # Write your own question here
# ===================

question = your_question if your_question else questions.get(choice, questions["A"])

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

## Complex Multi-Table Analytics

These queries require joining 3+ tables, using CTEs or subqueries, and computing derived metrics. They test Genie's ability to plan multi-step SQL.

In [ ]:
# Q42: Multi-table join (vehicles + orders + order_items + accessories + services)
question = "Top 5 vehicle models by total revenue including accessories and services"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

In [ ]:
# Q36: Quota comparison (salesperson_quotas + orders + employees)
question = "Which salespersons have exceeded their quota, and by how much?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

## Expert-Level Analytics

Window functions, multi-CTE chains, and full pipeline analysis.

Complexity does not equal difficulty for the AI. A 7-CTE sales funnel is hard SQL, but the intent is clear. A question like "luxury vehicles" is simple SQL but ambiguous since there's no definition of "luxury" in the schema. Clear intent matters more than query complexity.

In [ ]:
# Q49: Full sales funnel, 7 CTEs from leads to deliveries
question = "Create a full sales funnel analysis from lead source to closed deal, showing conversion rates at each stage"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

## Where AI Struggles

These patterns tend to have higher failure rates. Since LLMs are stochastic, you may see correct answers on some runs and failures on others. Try running them multiple times.

Three patterns to watch for:

### a) Correct Refusal: Data Doesn't Exist

The AI *should* refuse when the data genuinely isn't available. "Horsepower" sounds reasonable for a car dealership, but our schema has no horsepower column. Genie is generally good at recognizing missing columns.

In [ ]:
# Q18: Asks about data that doesn't exist in schema
question = "What is the average horsepower of our vehicles?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)
print("\n--- What to observe ---")
print("Did it refuse? The vehicles table has make, model, year, MSRP but no")
print("horsepower column. A good response acknowledges the data isn't available.")
print("This pattern usually works well since Genie checks column existence.")

### b) Ambiguous Business Concept

"Luxury vehicles" sounds clear to a human, but there's no `is_luxury` flag or price threshold defined in the schema. The AI may fabricate a definition (e.g., MSRP > $50k) and return confident-looking but meaningless results, or it may ask for clarification. This is one of the harder failure patterns to catch because the answer *looks* plausible.

In [ ]:
# Q33: Ambiguous concept, no definition of "luxury" in schema
question = "What percentage of our inventory is luxury vehicles?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)
print("\n--- What to observe ---")
print("Did it invent a definition of 'luxury'? Ask for clarification? Refuse?")
print("There is no luxury classification in the schema. Any concrete answer")
print("required the AI to fabricate a threshold. Sometimes it asks you to define")
print("the term (good). Sometimes it silently invents one (dangerous).")

### c) Hallucination: General Knowledge Override

Instead of querying the database, the AI sometimes bypasses Genie entirely and answers from its general training knowledge. The result looks authoritative but contains no actual data from your tables. Watch for answers that list generic information (e.g., common Toyota models from the internet) rather than rows from the `vehicles` table. This is the hardest failure pattern to detect because the content seems plausible.

In [ ]:
# Q2: General knowledge hallucination risk
question = "List all available vehicles made by Toyota"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)
print("\n--- What to observe ---")
print("Did the answer come from the DATABASE or from GENERAL KNOWLEDGE?")
print("A correct response queries the vehicles table and returns specific rows.")
print("A hallucination lists well-known Toyota models (Corolla, Camry, RAV4...)")
print("from the LLM's training data with no SQL executed at all.")
print("This is the hardest failure to detect because the content seems plausible.")

## Activity 2: Stump the AI

Try to write a question that exposes a limitation. Some hints:
- Business concepts that have no direct column mapping (e.g., "profitable", "at-risk", "high-performing")
- Ambiguous terms with multiple interpretations (e.g., "recent", "best", "active")
- Cross-domain questions that seem simple but need information not in the schema

After running it, classify the response: Did the AI refuse correctly? Return meaningless data? Or surprise you with a good answer?

In [ ]:
# === WRITE YOUR QUESTION ===
your_adversarial_question = "Which of our vehicles are best sellers?"  # <-- Change this
# ===========================

print(f"Question: {your_adversarial_question}")
print("=" * 60)

response = supervisor.query(your_adversarial_question, reset_history=True)
print(response)

print("\n--- Your Classification ---")
print("[ ] Refused correctly (data doesn't exist)")
print("[ ] Returned meaningless/fabricated data")
print("[ ] Actually answered well (surprising!)")

## Document & Policy Queries: RAG Agent

Not all questions are about data. The RAG agent searches a vector index of company documents (policies, procedures, guidelines) and synthesizes answers grounded in source documents.

```
Question → Vector Search → Retrieve Top-K Chunks → LLM Synthesis → Answer + Sources
```

We'll first query through the **supervisor** (which routes and may rephrase), then compare with **direct RAG access** to see the raw retrieval output.

In [ ]:
# Policy question, routes to RAG agent
question = "What warranty coverage comes with certified pre-owned vehicles?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

In [ ]:
# Compensation question, routes to RAG agent
question = "How does sales commission work at Velocity Motors?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

### Direct RAG Access

The supervisor routes to RAG and may rephrase the answer. Calling the RAG agent directly shows the raw retrieved documents and grounded answer before the supervisor synthesizes.

In [ ]:
# Direct RAG access, bypasses supervisor, shows raw retrieval
rag = RAGAgent(config)

# Same questions, direct access
for q in ["What warranty coverage comes with CPO vehicles?", "How does sales commission work?"]:
    print(f"Question: {q}")
    print("-" * 60)
    result = rag.query(q)
    if result.success:
        print(result.answer)
        print(f"\n{result.format_sources()}")
    else:
        print(f"Error: {result.error}")
    print("=" * 60)
    print()

## Combined Agent Query: Data + Documents

This is where multi-agent orchestration matters: a single question that requires *both* structured data and document knowledge. The supervisor routes to both agents and synthesizes the results.

In [ ]:
# Combined query, needs both Genie (top-selling models) and RAG (warranty info)
question = "What warranty coverage applies to our top-selling vehicle models?"

print(f"Question: {question}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)
print("\n--- What Happened ---")
print("The supervisor identified this needs BOTH data (top-selling models from Genie)")
print("and document knowledge (warranty details from RAG), then combined the results.")

## Activity 3: Guess the Route

You've seen all three patterns (Genie-only, RAG-only, and combined). Predict which agent handles each question before running it.

In [ ]:
# Pick a question (0-3) and predict the route before running
routing_questions = {
    0: ("What is the average discount by customer segment?", "Think: data or docs?"),
    1: ("How does our vehicle exchange policy work?", "Think: data or docs?"),
    2: ("Are fleet customers meeting their required service intervals?", "Think: one agent or both?"),
    3: ("What financing rates do we offer for certified pre-owned vehicles?", "Think: data or docs?"),
}

# === CHANGE THIS ===
pick = 0  # Choose 0-3
your_prediction = "Genie"  # Your prediction: "Genie", "RAG", or "Both"
# ===================

question, hint = routing_questions[pick]

print(f"Question: {question}")
print(f"Hint: {hint}")
print(f"Your prediction: {your_prediction}")
print("=" * 60)

response = supervisor.query(question, reset_history=True)
print(response)

print("\n--- Routing Key ---")
print("Q0: Genie (discount data is in orders/order_items)")
print("Q1: RAG (exchange policy is a document)")
print("Q2: Both (service data from Genie + service policy from RAG)")
print("Q3: RAG (financing rates are in policy documents)")

## Summary

**What we covered:**
- Simple to complex data queries with progressive difficulty
- Where AI struggles: missing data refusals, ambiguous concepts, general knowledge hallucination
- Document/policy retrieval via RAG (supervisor-routed vs. direct access)
- Combined data + document queries
- Agent routing patterns

**Next steps:**
- **`02_multi_genie_orchestration.ipynb`**: Parallel queries across multiple Genie Spaces with report generation
- **`03_build_your_agent.ipynb`**: Build your own LangGraph agent from scratch

In [ ]:
# Clear conversation history
supervisor.clear_history()
print("Conversation history cleared.")